## Mini Proc Model with MiniAE to demonstrate the use of LTN in an Autoencoder
1. Generate the data
2. Train the Autoencoder
3. Modify the AE with LTN

## Mini Proc Model
### Process

e1 -> e2 -> e3 -> e4 -> e5

event attributes = name, user

event_names = ["Create SC", "Approve SC", "Create PO", "Approve PO", "Pay"]

user_names = ["Dev", "Chantal", "Seokju", "Jonas", "Kaly"]

### Valid traces
1. ["Create SC", "Approve SC", "Create PO", "Approve PO", "Pay"]
1. ["Create SC", "Create PO", "Approve SC", "Approve PO", "Pay"]

### Event User Mapping
1. "Create SC" : "Dev", "Chantal" 
2. "Approve SC" : "Kaly"
3. "Create PO" : "Dev", "Jonas"
4. "Approve PO" : "Kaly"
5. "Pay" : "Seokju"

### Data
1. Traces = 1000
2. p_anomaly = 0.3 # This means that the possibility that a given trace is anomalous is 0.3
3. Anomaly types:
    1. Control flow: irregular flow ordering
    2. Attribute: Wrong attributes assigned

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from pprint import pprint
import itertools
import pickle

np.random.seed(0)

In [2]:
import tensorflow as tf
from tensorflow import keras
import ltn
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU found")
    print("Memory growth set")
else:
    print("No GPU found")

2025-03-22 18:42:15.639745: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742665336.642324   71584 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742665336.889302   71584 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742665336.569879   71584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742665336.569916   71584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742665336.569918   71584 computation_placer.cc:177] computation placer alr

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set


In [3]:
class Event(dict):
    """
    Event class with following keys:
    :name: name of the event
    :user: user who performed the event
    :case_id: case id of the event
    """
    pass
class Case(list):
    """
    Case is a list of traces
    """
    pass

In [11]:
class Dataset():
    def __init__(self, max_cases=None, anomaly_probabilty=None):
        self.event_names = set(["Create SC", "Approve SC", "Create PO", "Approve PO", "Pay"])
        self.user_names = set(["Dev", "Chantal", "Seokju", "Jonas", "Kaly"])
        self.valid_traces = [
            ["Create SC", "Approve SC", "Create PO", "Approve PO", "Pay"],
        ]
        self.invalid_traces_control_flow = [
            ["Approve SC", "Create SC", "Create PO", "Approve PO", "Pay"],
            ["Create SC", "Approve SC", "Approve PO", "Create PO", "Pay"],
            ["Create SC", "Approve SC", "Create PO", "Pay", "Approve PO"],            
            ["Create PO", "Approve PO", "Create SC", "Approve SC", "Pay"]
        ]
        self.event_user_mapping = {
            "Create SC": ["Dev", "Chantal"],
            "Approve SC": ["Kaly"],
            "Create PO": ["Dev", "Jonas"],
            "Approve PO": ["Kaly"],
            "Pay": ["Seokju"]
        }
        self.event_user_mapping_inv = {
            "Dev": ["Create SC", "Create PO"],
            "Chantal": ["Create SC"],
            "Kaly": ["Approve SC", "Approve PO"],
            "Jonas": ["Create PO"],
            "Seokju": ["Pay"]
        }
        self.anomaly_probability = 0.3
        self.max_cases = 1000
        self.actual_cases = 0
        if max_cases is not None:
            self.max_cases = max_cases
        if anomaly_probabilty is not None:
            self.anomaly_probability = anomaly_probabilty
        self.raw_dataset = []
        self.case_id_counter = 0
        self.encoders = {}
        self.encoders["name"] = LabelEncoder()
        self.encoders["name"].fit(list(self.event_names))
        print(f"encoders[name]:\n{[(str(c), str(t)) for c, t in list(zip(self.encoders['name'].classes_, self.encoders['name'].transform(self.encoders['name'].classes_)))]}")
        self.encoders["user"] = LabelEncoder()
        self.encoders["user"].fit(list(self.user_names))
        print(f"encoders[user]:\n{[(str(c), str(t)) for c, t in list(zip(self.encoders['user'].classes_, self.encoders['user'].transform(self.encoders['user'].classes_)))]}")
        
        
    def create_valid_traces(self, num_traces):
        partial_dataset = []
        cases_per_trace = num_traces // len(self.valid_traces)
        for valid_trace in self.valid_traces:
            for _ in range(cases_per_trace):
                case = Case()
                event = Event()
                for event_name in valid_trace:
                    event["name"] = event_name
                    event["user"] = np.random.choice(self.event_user_mapping[event_name])
                    event["case_id"] = self.case_id_counter
                    # case.append(event.copy())
                    partial_dataset.append(event.copy())
                self.case_id_counter += 1
                # partial_dataset.append(case)
        return partial_dataset, len(partial_dataset)
                
    def create_invalid_traces_control_flow(self, num_traces):
        partial_dataset = []
        cases_per_trace = num_traces // len(self.invalid_traces_control_flow)
        for invalid_trace in self.invalid_traces_control_flow:
            for _ in range(cases_per_trace):
                case = Case()
                event = Event()
                for event_name in invalid_trace:
                    event["name"] = event_name
                    event["user"] = np.random.choice(self.event_user_mapping[event_name])
                    event["case_id"] = self.case_id_counter
                    # case.append(event.copy())
                    partial_dataset.append(event.copy())
                self.case_id_counter += 1
                # partial_dataset.append(case)
        return partial_dataset, len(partial_dataset)
    
    def create_invalid_traces_attribute(self, num_traces):
        partial_dataset = []
        cases_per_trace = num_traces // len(self.valid_traces)
        for valid_trace in self.valid_traces:
            for _ in range(cases_per_trace):
                case = Case()
                event = Event()
                for event_name in valid_trace:
                    event["name"] = event_name
                    wrong_users = set(self.user_names) - set(self.event_user_mapping[event_name])
                    event["user"] = np.random.choice(list(wrong_users))
                    event["case_id"] = self.case_id_counter
                    # case.append(event.copy())
                    partial_dataset.append(event.copy())
                self.case_id_counter += 1
                # partial_dataset.append(case)
        return partial_dataset, len(partial_dataset)
    
    def create_dataset(self, max_cases=None, anomaly_probabilty=None):
        if max_cases is None:
            max_cases = self.max_cases
        else:
            self.max_cases = max_cases
        if anomaly_probabilty is None:
            anomaly_probabilty = self.anomaly_probability
        else:
            self.anomaly_probability = anomaly_probabilty
        num_anomalous_cases = int(max_cases * anomaly_probabilty)
        num_normal_cases = max_cases - num_anomalous_cases
        raw_dataset, actual_count = self.create_valid_traces(num_normal_cases)
        self.raw_dataset += raw_dataset
        self.actual_cases += actual_count
        raw_dataset, actual_count = self.create_invalid_traces_control_flow(num_anomalous_cases // 2)
        self.raw_dataset += raw_dataset
        self.actual_cases += actual_count
        raw_dataset, actual_count = self.create_invalid_traces_attribute(num_anomalous_cases // 2)
        self.raw_dataset += raw_dataset
        self.actual_cases += actual_count
    
    @property
    def raw_dataset_as_df(self):
        # name, user, case_id"
        return pd.DataFrame(self.raw_dataset)
    
    @property
    def raw_dataset_as_np_array(self):
        np_df = self.raw_dataset_as_df.to_numpy().reshape(-1, 5, 3)
        return np_df
    
    @property
    def encoded_features(self):
        without_case_id = self.raw_dataset_as_np_array[:, :, 0:2] # name, user
        # without_case_id_one_row_one_case = without_case_id.reshape(-1, 10)
        without_case_id_one_row_one_event = without_case_id.reshape(-1, 2)
        just_names = without_case_id_one_row_one_event[:, 0]
        just_users = without_case_id_one_row_one_event[:, 1]
        encoded_names = self.encoders["name"].transform(just_names).reshape(-1, 1)
        encoded_users = self.encoders["user"].transform(just_users).reshape(-1, 1)
        encoded_data = np.hstack((encoded_names, encoded_users)).reshape(-1, 10)
        return encoded_data
    
    @property
    def one_hot_encoded_features(self):
        one_row_one_event = self.encoded_features.reshape(-1, 2)
        from sklearn.preprocessing import OneHotEncoder

        name_column = one_row_one_event[:, 0].reshape(-1, 1)  # Get name column
        user_column = one_row_one_event[:, 1].reshape(-1, 1)  # Get user column

        self.one_hot_encoder = {}
        self.one_hot_encoder["name"] = OneHotEncoder(sparse_output=False)
        self.one_hot_encoder["user"]= OneHotEncoder(sparse_output=False)

        one_hot_encoded_name = self.one_hot_encoder["name"].fit_transform(name_column).reshape(-1, 5)
        one_hot_encoded_user = self.one_hot_encoder["user"].fit_transform(user_column).reshape(-1, 5)
        
        one_hot_encoded_data = np.hstack((one_hot_encoded_name, one_hot_encoded_user)).reshape(-1, 50)
        return one_hot_encoded_data
    
    @property
    def one_hot_encoded_features_2d(self):
        return self.one_hot_encoded_features.reshape(-1, 10, 5)
    
    def inverse_one_hot_encoded_features_to_int(self, one_hot_encoded_data):
        """
        There are 50 columns in a completely one hot encoded data
        thats 5 events and 5 users altenately
        first we deconde to integer level
        this function is just for one vector but maybe can be broadcasted over the whole dataset
        """
        one_row_one_event = one_hot_encoded_data.reshape(-1, 10)
        name_column = one_row_one_event[:, 0:5].reshape(-1, 5)
        user_column = one_row_one_event[:, 5:10].reshape(-1, 5)
        de_encoded_name = self.one_hot_encoder["name"].inverse_transform(name_column).reshape(-1, 1)
        de_encoded_user = self.one_hot_encoder["user"].inverse_transform(user_column).reshape(-1, 1)
        de_encoded_data = np.hstack((de_encoded_name, de_encoded_user)).reshape(-1, 2)
        return de_encoded_data.reshape(-1, 10)

    def inverse_one_hot_encoded_features_to_string(self, one_hot_encoded_data):
        """
        There are 50 columns in a completely one hot encoded data
        thats 5 events and 5 users altenately
        first we deconde to integer level
        then to string level
        this function is just for one vector but maybe can be broadcasted over the whole dataset
        """
        one_hot_encoded_data_int = self.inverse_one_hot_encoded_features_to_int(one_hot_encoded_data)
        one_row_one_event = one_hot_encoded_data_int.reshape(-1, 2)
        name_column = one_row_one_event[:, 0]
        user_column = one_row_one_event[:, 1]
        de_encoded_name = self.encoders["name"].inverse_transform(name_column).reshape(-1, 1)
        de_encoded_user = self.encoders["user"].inverse_transform(user_column).reshape(-1, 1)
        de_encoded_data = np.hstack((de_encoded_name, de_encoded_user)).reshape(-1, 2)
        return de_encoded_data.reshape(-1, 10)
        
        
        
    pass
    
synth_dataset = Dataset(max_cases=1000, anomaly_probabilty=0.3)
synth_dataset.create_dataset()
one_hot_flat = synth_dataset.one_hot_encoded_features
print(f"one_hot_flat shape: {one_hot_flat.shape}")
print(f"Raw features test:\n{synth_dataset.raw_dataset_as_df[:5]}")
print(f"Encoded Features test:\n{synth_dataset.encoded_features[0]}")
# pprint(f"One hot Encoded feature test: {one_hot_flat[0]}")
print(f"De encoding test:\n{synth_dataset.inverse_one_hot_encoded_features_to_int(one_hot_flat[0])}")
# from pprint import pprint
# pprint(f"Encoded Features: {temp.shape}")
# temp = synth_dataset.one_hot_encoded_features
# pprint(f"One hot Encoded Features: {temp.shape}")
# int_de_encode = synth_dataset.inverse_one_hot_encoded_features_to_int(temp[0])
# print(f"De encoding test: {int_de_encode}")
# str_de_encode = synth_dataset.inverse_one_hot_encoded_features_to_string(temp[0])


encoders[name]:
[('Approve PO', '0'), ('Approve SC', '1'), ('Create PO', '2'), ('Create SC', '3'), ('Pay', '4')]
encoders[user]:
[('Chantal', '0'), ('Dev', '1'), ('Jonas', '2'), ('Kaly', '3'), ('Seokju', '4')]
one_hot_flat shape: (998, 50)
Raw features test:
         name     user  case_id
0   Create SC  Chantal        0
1  Approve SC     Kaly        0
2   Create PO      Dev        0
3  Approve PO     Kaly        0
4         Pay   Seokju        0
Encoded Features test:
[3 0 1 3 2 1 0 3 4 4]
De encoding test:
[[3 0 1 3 2 1 0 3 4 4]]


In [ ]:
def model_fn(dataset: Dataset):
    from tensorflow.keras.layers import Input, Dense, Dropout, GaussianNoise
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam

    hidden_layers = 2
    hidden_size_factor = .2
    noise = True

    features = dataset.one_hot_encoded_features

    # Parameters
    input_size = features.shape[1]

    # Input layer
    input = Input(shape=(input_size,), name='input')
    x = input

    # Noise layer
    if noise is not None:
        x = GaussianNoise(noise)(x)

    # Hidden layers
    for i in range(hidden_layers):
        if isinstance(hidden_size_factor, list):
            factor = hidden_size_factor[i]
        else:
            factor = hidden_size_factor
        x = Dense(int(input_size * factor), activation='relu', name=f'hid{i + 1}')(x)
        x = Dropout(0.5)(x)

    # Output layer
    output = Dense(input_size, activation='softmax', name='output')(x)

    # Build model
    model = Model(inputs=input, outputs=output)

    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.0001, beta_2=0.99),
        loss='mean_squared_error',
    )

    return model, features, features

In [6]:
dae_model, _, _ = model_fn(synth_dataset)

I0000 00:00:1742665368.314453   71584 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3586 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [7]:
dae_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_noise (GaussianNoise)  │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hid1 (Dense)                    │ (None, 10)             │           510 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hid2 (Dense)                    │ (None, 10)             │           110 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 50)             │           550 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,170 (4.57 KB)

 Trainable params: 1,170 (4.57 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = dae_model.fit(
    one_hot_flat, one_hot_flat,
    epochs=30,
    batch_size=100,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

Epoch 1/30


I0000 00:00:1742665420.832303   71790 service.cc:152] XLA service 0x7f9da800b140 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1742665420.832344   71790 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2025-03-22 18:43:41.064077: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1742665421.449886   71790 cuda_dnn.cc:529] Loaded cuDNN version 90300


1/8 ━━━━━━━━━━━━━━━━━━━━ 2:55 25s/step - loss: 0.1926

I0000 00:00:1742665423.046589   71790 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


8/8 ━━━━━━━━━━━━━━━━━━━━ 58s 5s/step - loss: 0.1927 - val_loss: 0.1923
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 29s 4s/step - loss: 0.1926 - val_loss: 0.1923
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 29s 4s/step - loss: 0.1927 - val_loss: 0.1922
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 31s 4s/step - loss: 0.1924 - val_loss: 0.1922
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 28s 4s/step - loss: 0.1926 - val_loss: 0.1922
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 29s 4s/step - loss: 0.1923 - val_loss: 0.1922
Epoch 7/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 33s 4s/step - loss: 0.1924 - val_loss: 0.1921
Epoch 8/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - loss: 0.1924 - val_loss: 0.1921
Epoch 9/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 28s 4s/step - loss: 0.1921 - val_loss: 0.1921
Epoch 10/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 28s 4s/step - loss: 0.1921 - val_loss: 0.1921
Epoch 11/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 26s 3s/step - loss: 0.1919 - val_loss: 0.1920
Epoch 12/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - loss: 0.1920 - val_loss: 0.1920
Epoch 13/30
8/8 ━━━━━━━━

In [9]:
test_vector = one_hot_flat[0].reshape(1, -1)
pprint(test_vector)
op = dae_model.predict(test_vector)
pprint(op)

array([[0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
        0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
        0., 1.]])
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
array([[0.0166067 , 0.02100558, 0.02844806, 0.04393214, 0.01816451,
        0.01856961, 0.0292376 , 0.02650399, 0.01257782, 0.01521956,
        0.01342703, 0.02358253, 0.02030783, 0.01506363, 0.01267774,
        0.01632891, 0.01598162, 0.01708384, 0.01604253, 0.01728806,
        0.01092978, 0.01516455, 0.03138547, 0.02828698, 0.02063006,
        0.0263239 , 0.01419331, 0.02460706, 0.01878215, 0.01097424,
        0.03118865, 0.02393547, 0.02679867, 0.02102983, 0.01077907,
        0.01044197, 0.01583627, 0.02705074, 0.04687871, 0.02073128,
        0.01642276, 0.01884731, 0.01913829, 0.01861965, 0.02285407,
        0.02058712, 0.00947994, 0.01431353, 0.01559206, 0.01014772]],
      dtype=float32)


In [10]:
print(op.shape)

(1, 50)


In [28]:
def model_2d_fn(dataset: Dataset):
    """
    Two dimentional variational autoencoder
    """
    from tensorflow.keras.layers import Input, Dense, Dropout, GaussianNoise, Reshape, Flatten, Concatenate
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam

    hidden_layers = 2
    hidden_size_factor = .2
    noise = True

    features = dataset.one_hot_encoded_features_2d

    # Parameters
    input_size_1 = features.shape[1]
    input_size_2 = features.shape[2]
    

    # Input layer
    input = Input(shape=(input_size_1, input_size_2, ), name='input')
    flat_input = Flatten()(input)
    x = flat_input

    # Noise layer
    if noise is not None:
        x = GaussianNoise(noise)(x)

    # Hidden layers
    for i in range(hidden_layers):
        if isinstance(hidden_size_factor, list):
            factor = hidden_size_factor[i]
        else:
            factor = hidden_size_factor
        x = Dense(int(input_size_1 * input_size_2 * factor), activation='relu', name=f'hid{i + 1}')(x)
        x = Dropout(0.5)(x)

    # Output layer
    outputs = []
    for i in range(input_size_1):
        output = Dense(input_size_2, activation='softmax', name=f'output_{i}')(x)
        outputs.append(output)

    outputs = Concatenate(axis=1)(outputs)
    outputs = Reshape((input_size_1, input_size_2))(outputs)
    # Build model
    model = Model(inputs=input, outputs=outputs)

    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.0001, beta_2=0.99),
        loss='mean_squared_error',
    )

    return model, features, features

In [29]:
dae_model_2d, _, _ = model_2d_fn(synth_dataset)
dae_model_2d.summary()


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 10, 5)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 50)        │          0 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise_8    │ (None, 50)        │          0 │ flatten_2[0][0]   │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hid1 (Dense)        │ (None, 10)        │        510 │ gaussian_noise_8… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 10)        │          0 │ hid1[0][0]        │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hid2 (Dense)        │ (None, 10)        │        110 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 10)        │          0 │ hid2[0][0]        │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_0 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_1 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_2 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_3 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_4 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_5 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_6 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_7 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_8 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_9 (Dense)    │ (None, 5)         │         55 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 50)        │          0 │ output_0[0][0],   │
│ (Concatenate)       │                   │            │ output_1[0][0],   │
│                     │                   │            │ output_2[0][0],   │
│                     │                   │            │ output_3[0][0],   │
│                     │                   │            │ output_4[0][0],   │
│                     │                   │            │ output_5[0][0],   │
│                     │                   │            │ output_6[0][0],   │
│                     │                   │            │ output_7[0][0],   │
│                     │                   │            │ output_8[0][0],   │
│                     │                   │            │ output_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 10, 5)     │          0 │ concatenate[0][0

 Total params: 1,170 (4.57 KB)

 Trainable params: 1,170 (4.57 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history_2d = dae_model_2d.fit(
    synth_dataset.one_hot_encoded_features_2d, synth_dataset.one_hot_encoded_features_2d,
    epochs=30,
    batch_size=100,
    shuffle=True,
    validation_split=0.2,
    verbose=1,
)

Epoch 1/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 123s 6s/step - loss: 0.1903 - val_loss: 0.1616
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - loss: 0.1876 - val_loss: 0.1615
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 31s 4s/step - loss: 0.1872 - val_loss: 0.1613
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 39s 5s/step - loss: 0.1867 - val_loss: 0.1611
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 36s 5s/step - loss: 0.1870 - val_loss: 0.1609
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 29s 4s/step - loss: 0.1838 - val_loss: 0.1608
Epoch 7/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 39s 5s/step - loss: 0.1835 - val_loss: 0.1606
Epoch 8/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - loss: 0.1866 - val_loss: 0.1604
Epoch 9/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - loss: 0.1842 - val_loss: 0.1603
Epoch 10/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - loss: 0.1830 - val_loss: 0.1601
Epoch 11/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 29s 4s/step - loss: 0.1811 - val_loss: 0.1600
Epoch 12/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 27s 4s/step - loss: 0.1786 - val_loss: 0.1598
Epoch 13/30


In [36]:
test_vector_2d = synth_dataset.one_hot_encoded_features_2d[0].reshape(-1, 10, 5)
pprint(test_vector_2d)
op = dae_model_2d.predict(test_vector_2d)
pprint(op)

array([[[0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1.]]])
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
array([[[0.19383162, 0.1942565 , 0.20027354, 0.21217172, 0.19946665],
        [0.20670557, 0.20359246, 0.19670336, 0.19876458, 0.19423404],
        [0.19443034, 0.210501  , 0.2021707 , 0.1934097 , 0.1994883 ],
        [0.19584723, 0.19331126, 0.20012146, 0.20957144, 0.20114864],
        [0.20242696, 0.19981435, 0.20594789, 0.19730157, 0.19450915],
        [0.19314009, 0.20439652, 0.20900348, 0.19748254, 0.19597735],
        [0.21157958, 0.20013109, 0.19689135, 0.19696957, 0.1944284 ],
        [0.1949769 , 0.19727586, 0.19761239, 0.21181425, 0.19832055],
        [0.19459526, 0.20361294, 0.19618669, 0.1936003 , 0.21200477],
        [0.19818817, 0.1949731